# LightOnOCR-2 Magyar Fine-tuning (v2)

**Futtatás előtt:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# 1. Telepítés
!pip install -q transformers>=4.45.0 peft datasets accelerate pillow

In [ ]:
# 2. Tanító adatok generálása
import json
import random
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

HUNGARIAN_WORDS = [
    "őr", "őriz", "ők", "ősz", "ősi", "őszinte", "őrült",
    "erő", "idő", "mező", "tető", "fő", "nő", "bő", "hő",
    "belső", "külső", "felső", "alsó", "utolsó", "első",
    "költő", "festő", "vezető", "Győr", "tükörfúrógép",
    "Csatornadíj", "vízdíj", "díj", "dőlt", "dől",
    "űr", "űrlap", "gyűrű", "tűz", "fűz", "gyűjt",
    "hűtő", "hűvös", "hűség", "szürke", "szűk",
    "halványszürke", "fizetendő", "összeg", "adószám",
    "Árvíztűrő", "ÁRVÍZTŰRŐ",
]

def generate_text():
    words = random.sample(HUNGARIAN_WORDS, min(8, len(HUNGARIAN_WORDS)))
    lines = [" ".join(words)]
    lines.append(f"Fizetendő összeg: {random.randint(1,99)} {random.randint(100,999):03d} Ft")
    lines.append(f"Csatornadíj: {random.randint(1,9)} {random.randint(100,999):03d} Ft")
    lines.append(f"Adószám: {random.randint(10000000,99999999)}-{random.randint(1,2)}-{random.randint(10,99)}")
    lines.append("öüóőúéáűí - ÖÜÓŐÚÉÁŰÍ")
    lines.append("Halványszürke szöveg, dőlt betűk")
    lines.append("Árvíztűrő tükörfúrógép - ÁRVÍZTŰRŐ TÜKÖRFÚRÓGÉP")
    return "\n".join(lines)

def render_text(text, width=700, font_size=22):
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", font_size)
    except:
        font = ImageFont.load_default()
    lines = text.split("\n")
    height = len(lines) * (font_size + 10) + 50
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    y = 25
    for line in lines:
        draw.text((25, y), line, fill="black", font=font)
        y += font_size + 10
    return img

# Generálás
Path("training_data/images").mkdir(parents=True, exist_ok=True)
annotations = []

NUM_SAMPLES = 200
for i in range(NUM_SAMPLES):
    text = generate_text()
    img = render_text(text, font_size=random.choice([20, 22, 24, 26]))
    img.save(f"training_data/images/{i:05d}.png")
    annotations.append({"image": f"{i:05d}.png", "text": text})

with open("training_data/annotations.jsonl", "w") as f:
    for a in annotations:
        f.write(json.dumps(a, ensure_ascii=False) + "\n")

print(f"✓ Generated {NUM_SAMPLES} training samples")

# Show example
from IPython.display import display
display(Image.open("training_data/images/00000.png"))

In [ ]:
# 3. Modell betöltése (BF16, 4-bit nélkül)
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "lightonai/LightOnOCR-2-1B-base"

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f"✓ Model loaded: {MODEL_ID}")
print(f"  Device: {model.device}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 4. LoRA konfiguráció
from peft import LoraConfig, get_peft_model

# Find target modules
target_modules = []
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        if any(x in name for x in ["q_proj", "k_proj", "v_proj", "o_proj"]):
            # Get the short name
            short_name = name.split(".")[-1]
            if short_name not in target_modules:
                target_modules.append(short_name)

print(f"Target modules: {target_modules}")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=target_modules if target_modules else ["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 5. Dataset előkészítése
import json
from datasets import Dataset
from PIL import Image

def load_data():
    data = []
    with open("training_data/annotations.jsonl") as f:
        for line in f:
            entry = json.loads(line)
            data.append({
                "image_path": f"training_data/images/{entry['image']}",
                "text": entry["text"]
            })
    return Dataset.from_list(data)

def process_example(example):
    image = Image.open(example["image_path"]).convert("RGB")
    
    # Simple format: image + text
    text = example["text"]
    
    # Process image
    image_inputs = processor.image_processor(image, return_tensors="pt")
    
    # Tokenize text
    text_inputs = processor.tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        max_length=512,
        truncation=True,
    )
    
    return {
        "pixel_values": image_inputs["pixel_values"].squeeze(0),
        "input_ids": text_inputs["input_ids"].squeeze(0),
        "attention_mask": text_inputs["attention_mask"].squeeze(0),
        "labels": text_inputs["input_ids"].squeeze(0),
    }

dataset = load_data()
print(f"Loaded {len(dataset)} examples")

processed_dataset = dataset.map(
    process_example,
    remove_columns=dataset.column_names,
    num_proc=1,
)
print(f"✓ Dataset processed: {len(processed_dataset)} examples")

In [ ]:
# 6. Training
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./lighton-hun-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    bf16=True,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset,
)

print("Starting training...")
trainer.train()

In [ ]:
# 7. Mentés
print("Saving LoRA adapter...")
model.save_pretrained("./lighton-hun-lora")

print("Merging LoRA weights...")
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./lighton-hun-merged")
processor.save_pretrained("./lighton-hun-merged")

print("✓ Model saved to ./lighton-hun-merged")

In [ ]:
# 8. Gyors teszt
from PIL import Image
import torch

test_img = Image.open("training_data/images/00000.png")
display(test_img)

inputs = processor.image_processor(test_img, return_tensors="pt").to(merged_model.device)
inputs["input_ids"] = processor.tokenizer("", return_tensors="pt")["input_ids"].to(merged_model.device)

with torch.no_grad():
    outputs = merged_model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
    )

result = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\nOCR result:")
print(result)

In [ ]:
# 9. Letöltés
!zip -r lighton-hun-merged.zip lighton-hun-merged/

from google.colab import files
files.download("lighton-hun-merged.zip")

print("\n" + "="*50)
print("KÖVETKEZŐ LÉPÉS MAC-EN:")
print("="*50)
print("unzip lighton-hun-merged.zip")
print("mlx_vlm convert --hf-path lighton-hun-merged --mlx-path models/lighton-hun-mlx -q --q-bits 4")